# Titanic: date lipsă, features și predicția supraviețuirii

Similar cu notebook-ul NLP pe rețete, aici transformăm datele Titanic în features numerice
și antrenăm un model Scikit-learn care estimează **cât de probabil** e să fi supraviețuit.

**Ținta (y):** `Survived` — 1 = a supraviețuit, 0 = nu.

Coloane importante:
| Coloană | Tip | Observație |
|---------|-----|------------|
| `Pclass` | categorie ordonată | clasa biletelui (1 > 2 > 3) |
| `Sex` | categorie | strong predictor |
| `Age` | numeric | **multe valori lipsă** |
| `SibSp`, `Parch` | numeric | familie la bord |
| `Fare` | numeric | preț bilet |
| `Embarked` | categorie | port de îmbarcare (**puține lipsă**) |
| `Cabin` | text/categorie | **majoritar lipsă** |
| `Name` | text | putem extrage titlul (Mr, Mrs, Miss…) |

## 1. Încărcarea datelor

In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("titanic/train.csv")
test = pd.read_csv("titanic/test.csv")

print("train:", train.shape)
print("test:", test.shape)
train.head()

train: (891, 12)
test: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
train.info()

In [ ]:
print("Distribuție Survived:")
print(train["Survived"].value_counts())
print("\nRata de supraviețuire:", round(train["Survived"].mean(), 3))

---
## 2. Date lipsă (`NaN`) — ce înseamnă și ce facem

În pandas, o valoare lipsă e de obicei `NaN` (*Not a Number*).

Un model Scikit-learn **nu acceptă** `NaN` în features (cu excepția unor modele speciale).
Trebuie să decidem: **completăm**, **ștergem** sau **transformăm** coloana.

In [ ]:
missing = train.isna().sum().sort_values(ascending=False)
missing_pct = (train.isna().mean() * 100).round(1)

missing_df = pd.DataFrame({
    "nr_lipsă": missing,
    "procent_%": missing_pct,
})
missing_df[missing_df["nr_lipsă"] > 0]

### Strategii tipice

| Situație | Strategie | Exemplu pe Titanic |
|----------|-----------|--------------------|
| Puține lipsă pe o coloană categorică | completează cu **modul** (cea mai frecventă valoare) | `Embarked` (doar 2 lipsă) |
| Lipsă pe coloană numerică | completează cu **mediana** (sau media) | `Age`, `Fare` |
| Foarte multe lipsă | creează un flag „are valoare?” sau extrage un semnal simplu; nu te baza pe valoarea brută | `Cabin` (~77% lipsă) |
| Rând aproape gol / irelevant | șterge rândul (`dropna`) | rar aici; pierdem date |
| Lipsă e informativă | păstrează info ca feature (ex. `Age_missing = 1`) | uneori vârsta lipsea sistematic |

### 2.1 Detectare: `isna`, `notna`, filtrare

In [ ]:
# Unde lipsește Age?
train.loc[train["Age"].isna(), ["Name", "Sex", "Pclass", "Age", "Survived"]].head()

In [ ]:
# Unde lipsește Embarked? (doar 2 pasageri)
train.loc[train["Embarked"].isna(), ["PassengerId", "Name", "Sex", "Pclass", "Fare", "Embarked"]]

### 2.2 Completare manuală cu pandas

- **mediană** pentru numerice (robustă la outlier-e față de medie)
- **mod** pentru categorii

In [ ]:
df = train.copy()

# Flag: lipsea vârsta? (informație utilă uneori)
df["Age_missing"] = df["Age"].isna().astype(int)

# Completare Age cu mediana pe tot setul
age_median = df["Age"].median()
df["Age"] = df["Age"].fillna(age_median)
print(f"Age completat cu mediana = {age_median}")
print("Age lipsă după fillna:", df["Age"].isna().sum())

In [ ]:
# Varianta mai bună: mediana pe grup (Sex + Pclass)
df2 = train.copy()
df2["Age"] = df2.groupby(["Sex", "Pclass"])["Age"].transform(
    lambda s: s.fillna(s.median())
)
print("Age lipsă după fillna pe grup:", df2["Age"].isna().sum())
df2.groupby(["Sex", "Pclass"])["Age"].median()

In [ ]:
# Embarked: modul (cea mai frecventă valoare)
embarked_mode = df["Embarked"].mode()[0]
df["Embarked"] = df["Embarked"].fillna(embarked_mode)
print(f"Embarked completat cu modul = '{embarked_mode}'")
print("Embarked lipsă după fillna:", df["Embarked"].isna().sum())

### 2.3 `Cabin`: prea multe lipsă — nu completăm „orbește”

Completarea a 77% din valori cu o constantă inventează date.
Mai util: **are cabină?** și/sau **litera punții** (A, B, C…).

In [ ]:
df["HasCabin"] = df["Cabin"].notna().astype(int)
df["Deck"] = df["Cabin"].str[0]  # prima literă; NaN rămâne NaN
df["Deck"] = df["Deck"].fillna("Unknown")

print(df["HasCabin"].value_counts())
print("\nDeck:")
print(df["Deck"].value_counts())

### 2.4 Completare cu Scikit-learn: `SimpleImputer`

În pipeline-uri ML, preferăm `SimpleImputer` — se învață pe train și se aplică pe test
(evită data leakage).

In [ ]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

age_imputed = num_imputer.fit_transform(train[["Age"]])
emb_imputed = cat_imputer.fit_transform(train[["Embarked"]])

print("Mediană învățată pentru Age:", num_imputer.statistics_[0])
print("Mod învățat pentru Embarked:", cat_imputer.statistics_[0])
print("Age NaN după imputer:", pd.isna(age_imputed).sum())

### 2.5 Ce **nu** facem (de obicei)

- `dropna()` pe tot tabelul doar pentru că `Cabin` e goală → pierzi majoritatea pasagerilor
- completezi `Age` pe **tot** dataset-ul (train+test) înainte de split → leakage
- ignori lipsurile și lași `NaN` în model → eroare la `fit`

In [ ]:
# Demonstrație: dropna pe Cabin ar distruge setul
print("Rânduri totale:", len(train))
print("După dropna(subset=['Cabin']):", len(train.dropna(subset=["Cabin"])))
print("După dropna() pe toate coloanele:", len(train.dropna()))

---
## 3. Feature engineering pe text / categorii

Ca la rețete (`name`, `tags`), extragem semnal din `Name` și encodăm categoriile.

### 3.1 Titlul din `Name` (Mr, Mrs, Miss…)

In [ ]:
df["Title"] = df["Name"].str.extract(r",\s*([^.]+)\.", expand=False)
print(df["Title"].value_counts().head(15))

# Grupăm titlurile rare
rare_titles = df["Title"].value_counts()
rare = rare_titles[rare_titles < 10].index
df["Title"] = df["Title"].replace(rare, "Rare")
df["Title"].value_counts()

### 3.2 Familie și singur la bord

In [ ]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
df[["SibSp", "Parch", "FamilySize", "IsAlone", "Survived"]].head()

### 3.3 Encodări (ca în notebook-ul NLP)

| Metodă | Coloane tipice |
|--------|----------------|
| One-Hot (`get_dummies` / `OneHotEncoder`) | `Sex`, `Embarked`, `Title`, `Deck` |
| Ordinal | `Pclass` (deja 1/2/3) |
| Label Encoding | rareori necesar aici |

In [ ]:
# One-Hot cu pandas
sex_ohe = pd.get_dummies(df["Sex"], prefix="Sex")
emb_ohe = pd.get_dummies(df["Embarked"], prefix="Emb")
title_ohe = pd.get_dummies(df["Title"], prefix="Title")

pd.concat([df[["Name", "Sex", "Embarked"]], sex_ohe, emb_ohe], axis=1).head(3)

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
sex_sk = ohe.fit_transform(df[["Sex"]])
print("Categorii Sex:", ohe.categories_)
print("Shape OneHot Sex:", sex_sk.shape)

# Pclass e deja ordinal (1=cea mai „scumpă")
ord_enc = OrdinalEncoder(categories=[[1, 2, 3]])
df["Pclass_ord"] = ord_enc.fit_transform(df[["Pclass"]])
df[["Pclass", "Pclass_ord"]].drop_duplicates().sort_values("Pclass")

---
## 4. Model Scikit-learn: probabilitatea de a supraviețui

Folosim un `Pipeline` care:
1. completează lipsurile (`SimpleImputer`)
2. scalează numericele
3. face One-Hot pe categorii
4. antrenează `LogisticRegression`

`predict_proba` → **probabilitatea** de Survived=1.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score,
)

def prepare_features(raw: pd.DataFrame) -> pd.DataFrame:
    """Feature engineering comun train/test (fără a atinge Survived)."""
    out = raw.copy()
    out["Age_missing"] = out["Age"].isna().astype(int)
    out["HasCabin"] = out["Cabin"].notna().astype(int)
    out["Deck"] = out["Cabin"].str[0].fillna("Unknown")
    out["Title"] = out["Name"].str.extract(r",\s*([^.]+)\.", expand=False)
    # Mapare titluri rare (aceleași reguli pe train și test)
    title_map = {
        "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
        "Lady": "Rare", "Countess": "Rare", "Capt": "Rare", "Col": "Rare",
        "Don": "Rare", "Dr": "Rare", "Major": "Rare", "Rev": "Rare",
        "Sir": "Rare", "Jonkheer": "Rare", "Dona": "Rare",
    }
    out["Title"] = out["Title"].replace(title_map)
    out.loc[~out["Title"].isin(["Mr", "Mrs", "Miss", "Master", "Rare"]), "Title"] = "Rare"
    out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    return out


data = prepare_features(train)

numeric_features = ["Age", "Fare", "SibSp", "Parch", "FamilySize", "Age_missing", "HasCabin", "IsAlone"]
categorical_features = ["Pclass", "Sex", "Embarked", "Title", "Deck"]

X = data[numeric_features + categorical_features]
y = data["Survived"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape, "| X_val:", X_val.shape)
X_train.head(3)

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

clf = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

clf.fit(X_train, y_train)
print("Model antrenat.")

### 4.1 Predicție clasă vs probabilitate

In [ ]:
y_pred = clf.predict(X_val)
y_proba = clf.predict_proba(X_val)[:, 1]  # P(Survived=1)

results = X_val.copy()
results["Survived_real"] = y_val.values
results["Survived_pred"] = y_pred
results["P_survive"] = y_proba.round(3)

results[["Sex", "Pclass", "Age", "Fare", "Survived_real", "Survived_pred", "P_survive"]].head(10)

In [ ]:
print("Accuracy:", round(accuracy_score(y_val, y_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_val, y_proba), 3))
print("\nClassification report:")
print(classification_report(y_val, y_pred, target_names=["Died", "Survived"]))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_val, y_pred))

In [ ]:
cv = cross_val_score(clf, X, y, cv=5, scoring="accuracy")
print("CV accuracy (5-fold):", np.round(cv, 3))
print("Media:", round(cv.mean(), 3), "±", round(cv.std(), 3))

cv_auc = cross_val_score(clf, X, y, cv=5, scoring="roc_auc")
print("CV ROC-AUC:", np.round(cv_auc, 3), "| media:", round(cv_auc.mean(), 3))

### 4.2 Alternativă: Random Forest

In [ ]:
rf = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        random_state=42,
        n_jobs=-1,
    )),
])

rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_val)[:, 1]
rf_pred = rf.predict(X_val)

print("RF Accuracy:", round(accuracy_score(y_val, rf_pred), 3))
print("RF ROC-AUC:", round(roc_auc_score(y_val, rf_proba), 3))
print(classification_report(y_val, rf_pred, target_names=["Died", "Survived"]))

---
## 5. Predicții pe `test.csv` (fără Survived)

Reantrenăm pe **tot** train-ul, apoi estimăm probabilitățile pe pasagerii din test.

In [ ]:
# Reantrenare pe tot train
final_model = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
final_model.fit(X, y)

test_feat = prepare_features(test)
X_test = test_feat[numeric_features + categorical_features]

# test are 1 Fare lipsă — SimpleImputer din pipeline o completează automat
print("Lipsă în X_test înainte de pipeline:")
print(X_test.isna().sum()[X_test.isna().sum() > 0])

test_pred = final_model.predict(X_test)
test_proba = final_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Name": test["Name"],
    "Sex": test["Sex"],
    "Pclass": test["Pclass"],
    "Survived": test_pred,
    "P_survive": np.round(test_proba, 3),
})
submission.head(10)

In [ ]:
# Exemple: cei mai „siguri” că ar fi supraviețuit / nu
print("Top 5 cea mai mare P_survive:")
print(submission.nlargest(5, "P_survive").to_string(index=False))

print("\nTop 5 cea mai mică P_survive:")
print(submission.nsmallest(5, "P_survive").to_string(index=False))

In [ ]:
# Salvare stil Kaggle (doar PassengerId + Survived)
kaggle_sub = submission[["PassengerId", "Survived"]]
kaggle_sub.to_csv("titanic/submission_sklearn.csv", index=False)
print("Salvat: titanic/submission_sklearn.csv")
kaggle_sub.head()

### Exemplu concret: cât de probabil e să rezisti?

In [ ]:
def survival_probability(model, example: dict) -> float:
    """Primește un dict cu feature-urile și întoarce P(Survived=1)."""
    row = pd.DataFrame([example])
    return float(model.predict_proba(row)[0, 1])


female_1st = {
    "Age": 28, "Fare": 80, "SibSp": 0, "Parch": 0, "FamilySize": 1,
    "Age_missing": 0, "HasCabin": 1, "IsAlone": 1,
    "Pclass": 1, "Sex": "female", "Embarked": "C", "Title": "Miss", "Deck": "B",
}
male_3rd = {
    "Age": 22, "Fare": 7.25, "SibSp": 0, "Parch": 0, "FamilySize": 1,
    "Age_missing": 0, "HasCabin": 0, "IsAlone": 1,
    "Pclass": 3, "Sex": "male", "Embarked": "S", "Title": "Mr", "Deck": "Unknown",
}

p1 = survival_probability(final_model, female_1st)
p2 = survival_probability(final_model, male_3rd)

print(f"Femeie, clasa 1, cabină:  P(survive) ≈ {p1:.1%}")
print(f"Bărbat, clasa 3, fără cabină: P(survive) ≈ {p2:.1%}")

---
## Rezumat

### Date lipsă
1. **Inspectează** cu `isna().sum()` / procent
2. **Age / Fare** → mediană (`fillna` sau `SimpleImputer`)
3. **Embarked** → mod (cea mai frecventă valoare)
4. **Cabin** → nu completa orbește; folosește `HasCabin` / `Deck`
5. Ideal: imputer **în Pipeline**, învățat doar pe train

### Encodări
- One-Hot: `Sex`, `Embarked`, `Title`, `Deck`
- Numerice scalate: `Age`, `Fare`, `FamilySize`…
- Feature-uri din text: titlul din `Name`

### Model
- `LogisticRegression` / `RandomForestClassifier`
- `predict` → 0/1
- `predict_proba[:, 1]` → **probabilitatea de a fi supraviețuit**